In [1]:
# Détection de fraude — EDA session 1 (6/08/2026)

#Dataset : IEEE-CIS Fraud Detection (Kaggle), `train_transaction.csv`

## Volumétrie
#- 590540 lignes × 394 colonnes

## Cible `isFraud`
#- Taux de fraude : 3.49 %
#- Conséquence : accuracy inutilisable car un modèle qui prédit « légitime » partout atteint 96,5 % d'accuracy en attrapant 0 fraude → métriques : precision / recall

## Types de colonnes
#- 380 numériques (float+int) / 14 texte

## Valeurs manquantes
#- 2 colonnes vides à >90 % (pires : colonne "dist2" vide à 93.6 % et colonne "D7" vide à 0.934099 )
#- Blocs V à taux de NaN identiques → vides sur les mêmes lignes → même source (blocs livrés par Vesta) → redondance possible à l'intérieur d'un bloc, piste de dégraissage
#- Décision : on ne supprime/remplit rien car (1) XGBoost traite nativement le format NaN (2) l'absence peut être corrélée à la fraude elle-même

In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("data/train_transaction.csv")

In [4]:
df.shape

(590540, 394)

In [5]:
df.sample(10)

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
4351,2991351,0,164700,48.950,W,9500,321.0,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
229962,3216962,0,5444232,2190.500,W,8949,555.0,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
549504,3536504,0,14507667,30.950,W,16132,111.0,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
209524,3196524,0,4836101,50.000,H,3983,583.0,150.0,visa,226.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3148,2990148,0,152851,80.000,W,9500,321.0,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
475932,3462932,0,12322451,117.000,W,12219,NaN,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
579667,3566667,0,15451381,85.000,S,15775,481.0,150.0,mastercard,102.0,...,22.0,0.0,146203.0,54536.0,0.0,2117.0,316.0,0.0,10065.0,2356.0
291625,3278625,0,7178416,107.950,W,6236,194.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
241031,3228031,0,5696107,78.667,C,4461,375.0,185.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
32840,3019840,0,829061,40.000,H,7585,553.0,150.0,visa,226.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
df["isFraud"].value_counts(normalize=True)

isFraud
0    0.96501
1    0.03499
Name: proportion, dtype: float64

In [7]:
df.dtypes.value_counts()
#df.select_dtypes(include="str").columns

float64    376
str         14
int64        4
Name: count, dtype: int64

In [8]:
nan_rates = df.isna().mean()
nan_rates.sort_values(ascending=False).head(20)

dist2    0.936284
D7       0.934099
D13      0.895093
D14      0.894695
D12      0.890410
D6       0.876068
D9       0.873123
D8       0.873123
V153     0.861237
V149     0.861237
V141     0.861237
V146     0.861237
V154     0.861237
V162     0.861237
V142     0.861237
V158     0.861237
V161     0.861237
V157     0.861237
V138     0.861237
V139     0.861237
dtype: float64

In [9]:
(nan_rates>0.9).sum()

np.int64(2)